In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

windspeed_df = pd.read_csv(str(ROOT / "data/DMI_ws.csv"))
irr_df = pd.read_csv(str(ROOT / "data/DMI_radiation.csv"))
PV_df = pd.read_csv(str(ROOT / "data/SyslabPV_15min_collective_PV.csv"))
PV_df["ts"] = pd.to_datetime(PV_df['ts'])
PV_df = PV_df.set_index("ts")
PV_df = PV_df
Wind_df = pd.read_csv(str(ROOT / "data/SyslabWind_15min_nozeros.csv"))
df = pd.read_csv(str(ROOT / "data/combined_data_positive_gen.csv"))

In [ ]:
seasonality = 24*4 
PV_df["Power diff"] = PV_df['Collective PV'].diff(periods=seasonality)

In [ ]:
result = seasonal_decompose(PV_df['Collective PV'].dropna(), model='addiditve', period=seasonality)
trend = result.trend.dropna()
seasonal = result.seasonal.dropna()
residual = result.resid.dropna()

In [ ]:

# Plot the decomposed components
plt.figure(figsize=(6,6))
#x_labels = [t.strftime('%m %Y') for t in PV_df.index]
plt.subplot(4, 1, 1)
plt.plot(PV_df.index,PV_df['Collective PV'], label='Original Series')

#plt.xticks(range(0, PV_df.size, 2880), x_labels[::2880], rotation=45)
plt.legend()

plt.subplot(4, 1, 2)
plt.plot(trend, label='Trend')
plt.legend()

plt.subplot(4, 1, 3)
plt.plot(seasonal, label='Seasonal')
plt.legend()

plt.subplot(4, 1, 4)
plt.plot(residual, label='Residuals')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
from statsmodels.tsa.stattools import adfuller
from pathlib import Path
import pmdarima as pm
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

In [ ]:
def plot_predicted_vs_measured(measured,predicted,dataset,exo,exo_name,plot=True):
    
    r2 = r2_score(measured, predicted)
    mae = mean_absolute_error(measured, predicted)
    rmse = mean_squared_error(measured, predicted) ** 0.5

    if plot: # Scatter plot: measured vs predicted on test set
        plt.figure(figsize=(6, 6))
        plt.scatter(measured, predicted, alpha=0.6)
        plt.xlabel("Measured power")
        plt.ylabel("Predicted power")
        plt.title(f"{dataset} model: measured vs predicted (test set)")

        min_val = min(measured.min(), predicted.min())
        max_val = max(measured.max(), predicted.max())
        plt.plot([min_val, max_val], [min_val, max_val])

        plt.show()
        plt.figure(figsize=(6, 6))
        plt.scatter(exo, predicted, alpha=0.6)
        plt.xlabel(f"{exo_name}")
        plt.ylabel("Predicted power")
        plt.title(f"{dataset} model: {exo_name} vs predicted (test set)")

        
        plt.show()

    
        print("Wind model")
        print("R²:", r2)
        print("MAE:", mae)
        print("RMSE:", rmse)
    
    return r2

In [ ]:
def Sarimax(Type,file_path,endo_name,exo_name,granularity=10,cut_in = 1 ,order = (1,0,1),seasonal_order = (1,1,1),gap = ["2025-06-14 00:00:00","2025-06-21 00:00:00"],save_new_file = False,filename="",plot = True):
    """Using the SARIMAX function to fill gaps in the endogenous variable using the exogeneous variable
    Granularity is the amount of time between measurements in minutes
    Type is either "Wind" or "Solar" """


    try:
        df = pd.read_csv(file_path,delimiter=",")
    except Exception as e:
        print(f"Error reading file: {e}")
        return
    df["ts"] = pd.to_datetime(df["ts"])
    df = df.set_index("ts") # Set datetime as index
    df = df.loc[df.index >"2025-05-14 07:00:00"] # start from when our power data starts


    df[exo_name] = df[exo_name].interpolate() # REMOVE ANY NANS

    df_modelled = df.copy()
    
    if Type == "Wind":
        # Wind power is proportional to the cubic wind speed
        df.loc[df[exo_name] < cut_in, endo_name] = 0 # set low wind periods to zero

        gap_measured = df.loc[(gap[0] < df.index) & (df.index < gap[1]),endo_name] # save the data that we will later use for testing
        df = df.loc[df[exo_name] >= cut_in] # remove low wind periods to reduce how much it gets skewed
        df.loc[(gap[0] < df.index )& (df.index < gap[1]),endo_name] = np.nan

        df["active"] = (df[exo_name]>= 3).astype(int)
        df["w2"] = df[exo_name]**2
        df["w3"] = df[exo_name]**3
        
        endo = df[endo_name]
        exo = df[[exo_name,"active","w3"]]
        print("Preparing Model for Wind Turbine")

    elif Type == "Solar":
        df.loc[df[exo_name] < cut_in, endo_name] = 0 # set night time to zero

        gap_measured = df.loc[(gap[0] < df.index) & (df.index < gap[1]),endo_name] # save the data that we will later use for testing
        
        df = df.loc[df[exo_name] >= cut_in] # remove night time to reduce how much it gets skewed
        df.loc[(gap[0] < df.index )& (df.index < gap[1]),endo_name] = np.nan

        endo = df[endo_name]
        exo = df[exo_name] 
        print("Preparing Model for PV unit")


    if gap_measured.isna().sum().astype(int) != 0: 
        print("Gap contains NaNs so the model can not be verified. Retry with different gap.")
        return 

    fraction_dayvsnight = df.size/df_modelled.size
    seasonality = 24*60/granularity*fraction_dayvsnight ####### Takes the average time between the time today and the same time tomorrow (complex since we removed nighttime values)
    seasonal_order_full = (*seasonal_order, int(seasonality))

    model = SARIMAX(
        endo,
        exog=exo,
        order=order,
        seasonal_order=seasonal_order_full,  # daily seasonality
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    results = model.fit()

    endo_modelled = results.get_prediction().predicted_mean # USES KALMAN SMOOTHING ( includes both past and future values to do "predictions")
    df["endo_filled"] = df[endo_name]
    df.loc[endo.isna(),"endo_filled"] = endo_modelled.loc[endo.isna()]



    # results.plot_diagnostics() # in case we want to see built in diagnostics 


    if Type == "Wind":        
        df_modelled.loc[df_modelled[exo_name] >= cut_in, endo_name +"modelled"] = df["endo_filled"] # redo the zero rows that were removed because low windspeed
        df_modelled.loc[df_modelled[exo_name] < cut_in, endo_name+ "modelled"] = 0 # set everything at with no windspeed
        # DO WE WANT THIS FOR WIND? 
    elif Type == "Solar":
        df_modelled.loc[df_modelled[exo_name] >= cut_in, endo_name +"modelled"] = df["endo_filled"] # redo the zero rows that were removed because irradiance was zero
        df_modelled.loc[df_modelled[exo_name] < cut_in, endo_name+ "modelled"] = 0 # set everything at night to 0
    
        ##### WE ARE FAKING HOW GOOD THE MODEL PERFORMS by how we set everyting less than cut in to zero in both the model and the original data
        ##### IDEALLY: only set to zero while creating the model and then afterwards verify the accuracy of it. 
        
    if save_new_file:
        file_name = f'{file_path[:-4]}_{filename}_{granularity}min__SARIMAX_{order}{seasonal_order}.csv'
        df_modelled.to_csv(file_name, index=True)

    
    gap_predicted = df_modelled.loc[(gap[0] < df_modelled.index) & (df_modelled.index < gap[1]),endo_name + "modelled"]
    gap_exo = df_modelled.loc[(gap[0] < df_modelled.index) & (df_modelled.index < gap[1]),exo_name]

    return plot_predicted_vs_measured(gap_measured,gap_predicted,endo_name,gap_exo,exo_name,plot)



In [ ]:
combined_data = str(ROOT / "data/combined_data_DMI_gen.csv")

SOLAR: 

for 60min resolution: 


(101,111) r2: 0.882 took 20 sec (30min res took 30 sec, 10min took over 12min)

(101,101) r2: 0.890

(101,100) r2: 0.890

(101,000) r2: 0.890

(001,000) r2: 0.892

(002,000) r2: 0.900

(102,000) r2: 0.886 # and tried different constellations but all reduced the r2



for 30min: 

(000,000) r2: 0.847

(001,000) r2: 0.865

(001,100) r2: 0.864

(001,101) r2: 0.864


for 10min: 

(000,000) r2: 0.796
(001,000) r2: 0.799
(001,001) r2: 0.790
(000,001) r2: 0.797

INVESTIGATE 60min (002,000) r2: 0.900 with different gaps

"2025-06-19 00:00:00","2025-06-21 00:00:00" : r2: 0.911

"2025-06-19 00:00:00","2025-06-29 00:00:00" r2: 0.877

"2025-06-19 00:00:00","2025-07-29 00:00:00" r2: 0.789

"2025-07-19 00:00:00","2025-07-29 00:00:00" r2: 0.707

"2025-07-19 00:00:00","2025-07-25 00:00:00" r2: 0.680

"2025-07-19 00:00:00","2025-07-20 00:00:00" r2: 0.645

"2025-07-05 00:00:00","2025-07-06 00:00:00" r2: 0.747

"2025-07-05 00:00:00","2025-07-20 00:00:00" r2: 0.720

"2025-10-05 00:00:00","2025-10-07 00:00:00" r2: 0.701

In [ ]:
gaps = [["2025-06-19 10:00:00","2025-06-22 15:00:00"], # edium
        ["2026-01-10 00:00:00","2026-01-21 00:00:00"], # long # CONTAINS NAN
        ["2025-10-29 00:00:00","2025-11-21 00:00:00"], # long # CONTAINS NAN
        ["2025-12-17 00:00:00","2025-12-18 12:00:00"], # short
        ["2025-05-19 00:00:00","2025-05-24 00:00:00"]] # medium

In [ ]:
gap = ["2025-06-19 00:00:00","2025-06-29 00:00:00"]
Sarimax("Solar",combined_data,"B715","Global Irradiance",granularity = 60,cut_in=1,order = (0,0,2),
                                seasonal_order = (0,0,0),save_new_file=False, filename="PV715 Modelled Data.csv",
                                gap = gap,plot = True)

In [ ]:
gaps = [["2025-12-17 00:00:00","2025-12-18 12:00:00"], # short
        ["2025-05-19 00:00:00","2025-05-24 00:00:00"]] # medium
for gap in gaps: 
        for order in orders: 
                for seasonal_order in seasonal_orders:
                        r2 = Sarimax("Solar",combined_data,"B715","Global Irradiance",granularity = res,order = order,
                                seasonal_order = seasonal_order,save_new_file=False, filename="PV715 Modelled Data.csv",
                                gap = gap,plot = False)
                        dic[(*order,*seasonal_order)].append(r2)
                print(f"Order {order} is completed")
        print(f"Time period: {gap} is completed")

In [ ]:
res = 60
combined_data = str(ROOT / f"data/combined_data_DMI_gen_{res}min.csv")

orders = [(0,0,0),(0,0,1),(0,0,2),(1,0,0),(2,0,0),(1,0,1),(2,0,1),(1,0,2),(2,0,2)]
seasonal_orders = [(0,0,0),(0,0,1),(0,1,1),(1,0,0),(1,1,0),(1,0,1),(1,1,1),(0,1,0)]
dic = {}
for order in orders: 
                for seasonal_order in seasonal_orders:
                        dic[(*order,*seasonal_order)] = [] 
for gap in gaps: 
        for order in orders: 
                for seasonal_order in seasonal_orders:
                        r2 = Sarimax("Solar",combined_data,"B715","Global Irradiance",granularity = res,order = order,
                                seasonal_order = seasonal_order,save_new_file=False, filename="PV715 Modelled Data.csv",
                                gap = gap,plot = False)
                        dic[(*order,*seasonal_order)].append(r2)
                print(f"Order {order} is completed")
        print(f"Time period: {gap} is completed")
        

# for Wind: 60min res

### looks weird (power prediction high when no power is produced)

(001,000) r2: 0.617

(002,000) r2: 0.584

(102,000) r2: 0.526

(002,100) r2: 0.595

(002,101) r2: 0.594

(002,010) r2: -0.12

(701,111): r2: 0.23

(002,201) r2: 0.596 

(002,102) r2: 0.596 

In [ ]:
gap = ["2025-06-19 00:00:00","2025-06-29 00:00:00"]
Sarimax("Wind",combined_data,"Aircon_WT Power","Windspeed",granularity = 60,cut_in=0.5,order = (0,0,1),
                                seasonal_order = (1,0,0),save_new_file=True, filename="Aircon.csv",
                                gap = gap,plot = True)

## Now lets try to model this again, but not only filling gaps, but doing backwards prediction! Try to fill Feb-May 2025